# Model Evaluation

## Agenda
- Cross validation
- Regularization
- Hyperparameter search
- More classification metrics
- Missing value imputation


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import urllib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


## Preparing the data


In [ ]:
url = 'https://pub-c88b3a7f2ed141418355b2bfb03c96e6.r2.dev/income.csv'
request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request) as response:
    df = pd.read_csv(response)


In [ ]:
df.sample(10)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
df_train, df_test = train_test_split(df, test_size=0.2, random_state=10)

## Missing values

Most algorithms don't work when missing values are present in the data. And in our dataset we have a few rows that contain missing values in some of the columns.

In [ ]:
# Count rows with at least one missing value
df_train.isna().any(axis=1).sum()

In [ ]:
df.info()

We don't just want to remove those rows, they're still useful! Also, missing data does not necessarily imply that there's something wrong with the row. 

Imputing missing values for categorical features is relatively simple - just treat the missing values as a separate category.

In [ ]:
df_train.value_counts("workclass", dropna=False)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Now we can use the imputer if needed for other missing values
missing_imputer = SimpleImputer(strategy="constant", fill_value="missing")
workclass_imputed = missing_imputer.fit_transform(df_train[["workclass"]])

# One-hot encode the imputed data
encoder = OneHotEncoder(handle_unknown="ignore")
workclass_encoded = encoder.fit_transform(workclass_imputed)

# continue with modelling

How can we impute continuous columns? All numerical columns in the dataset we're currently using have no missing data, so I'll create a fake feature with some data from a uniform distribution and some missing values.

In [ ]:
feature = np.random.uniform(size=50)
feature[:10] = np.nan

tmp = pd.DataFrame(feature, columns=["a"])

In [ ]:
mean_imputer = SimpleImputer(strategy="mean")
mean_imputer.fit_transform(tmp)[:20]

The feature was imputed with the mean of values in rows where data is not missing. "median" is another strategy we could have used here.

### Task

Fit a LogisticRegression model for predicting `income` using these features:
- `workclass`
- `hours_per_week`
- `relationship`
- `age`

For categorical variables, apply missing value imputation (strategy `constant`) and one-hot encoding. To continuous variables, apply PolynomialFeatures(degree=2) and StandardScaler.

Set penalty in LogisticRegression to "l1".

Use ColumnTransformer from last week to select which features to apply the transformations to.


## Putting it all together

Now we have all the tools that we need in order to successfully fit a classification or regression model.
- We know some algorithms (logistic regression, K-Nearest-Neighbor).
- We know some useful feature preprocessing steps:
  - One-hot encoding, that allows us to use categorical features in our models.
  - Polynomial transformations, that are useful for increasing the flexibility of the logistic regression model.
  - Feature scaling (minmax scaling and standard scaling), that improve the performance of most algorithms.
  - Missing value imputation.
- We know what regularization is and how it helps prevent overfitting.
- We know several metrics that allow us to measure how good our model performs.
- We know how to perform cross validation to validate how any specific model performs (in terms of the metric we chose), so as to choose the best model.
- We know what hyperparameters are and how to tune them with the help of cross validation.
- We know that we should keep a portion of our data for reporting the performance of our final model.

We can put all of these steps together to fit a very useful model. Let's do that step by step.


### Step 1

Fit a LogisticRegression model with penalty=None that predicts `income` from `hours_per_week` and `education_num`. Evaluate the model via cross-validation using roc_auc as the metric.

Note that we're beginning with something very basic - a simple algorithm with just two features that have no missing values.


### Step 2

Modify the code above by adding the `occupation` variable among the predictors. Keep in mind that it's a categorical variable with missing values. Use scikit-learn Pipeline and ColumnTransformer.


In [ ]:
# Hint

# import

# select features

categorical_transformer = Pipeline(
    steps=[
        ...
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, ["feat1", "feat2"]),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", ...),
    ]
)

# rest is the same

### Step 3

Modify the code above to apply PolynomialFeatures(degree=2) and StandardScaler to `hours_per_week` and `education_num` features.


### Step 4

Modify the code above by adding regularization to the model. Use l1 regularization, do not set a value for C. Remember to change the solver to "liblinear".


### Step 5

Modify the code above to find an optimal value for C by performing hyperparameter tuning. Use `GridSearchCV`.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"logreg__C": [10**i for i in range(-3, 3)]}
grid_search = GridSearchCV(...)
grid_search.fit(X_train, y_train)

print(...)


### Step 6

Add `native_country` feature to the model. Notice that this column has 41 categories, and that it contains missing values - apply missing value imputation and use max_categories=10 in OneHotEncoder.


scikit-learn is warning me about unknown categories in the data. This happens because during cross validation our validation set sometimes contains countries that are not in the training set. We can ignore this warning, since we set `handle_unknown="ignore"` in OneHotEncoder, which handles such cases.

I also elected to limit the number of categories to 10 biggest categories. OneHotEncoder puts the rest of the values in some "other" category.


### Step 7

Add these features to the model:
- `age`
- `race`

Apply the same transformations as we did so far for categorical and numerical features.

### Step 8

Find optimal values for C and class_weight between `None` and `"balanced"`.


### Step 9

Train a model with the best hyperparameters from the last step.

Report the performance of the model on the test dataset.




And that is what modelling looks like! We began with almost the simplest model possible and gradually added complexity while also making the model better. Then we reported the model's performance on test data.
